# 🧠 Pipeline: TinyML — Train → Export → Arduino C++
**End-to-end: Train a Perceptron, Export to C++, Deploy on Arduino**  
Diplomado Superior en Redes Neuronales Artificiales y Deep Learning

---
> Flujo completo de TinyML: creamos un dataset sintético de sensores, entrenamos un  
> perceptrón para clasificar, y exportamos el modelo a código C++ para Arduino.

---
## 🔄 Flujo de Trabajo TinyML

```
1. Recolectar datos (sensores) ──→ 2. Entrenar modelo ──→ 3. Exportar a C++ ──→ 4. Subir a Arduino
       ↑                                      ↓                         ↓
   Temperatura, luz,                      Perceptrón               código .ino
   humedad, etc.                          (2 features)             listo para IDE
```

**¿Por qué TinyML?**  
Ejecutar IA directamente en microcontroladores (sin WiFi, sin cloud, bajo consumo).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..')))
from libreria_modulo5 import *

import numpy as np
configurar_reproducibilidad(42)

---
## 🌡️ Dataset Sintético: Sensor de Temperatura y Luz

Simulamos un sistema que decide si un **ventilador (fan)** debe encenderse según:  
- **Temperatura**: 15–45 °C
- **Luz**: 0–1023 (como lecturas de un LDR)

**Regla**: Fan ON si `temp > 25` AND `luz > 500`.

In [ ]:
np.random.seed(42)
N = 200
temperatura = np.random.uniform(15, 45, N)
luz = np.random.uniform(0, 1023, N)

X = np.column_stack([temperatura, luz])
y = np.array([1 if t > 25 and l > 500 else 0 for t, l in X])

print(f"Dataset generado: {N} muestras")
print(f"  Clase 0 (fan OFF): {sum(y == 0)}")
print(f"  Clase 1 (fan ON):  {sum(y == 1)}")
print(f"\nPrimeras 5 filas:")
print("  Temperatura  Luz     Fan")
for i in range(5):
    print(f"  {X[i,0]:8.1f}  {X[i,1]:6.0f}  {y[i]}")

---
## 🧠 Entrenar Perceptrón

Usamos la regla de Rosenblatt (aprendizaje por corrección de error) para  
encontrar los pesos W y el sesgo b que separan las clases.

In [ ]:
res = perceptron_entrenar(X, y, lr=0.001, epochs=100, verbose=True)
print(f"\nPesos finales: W = {np.round(res['W'], 4)}")
print(f"Bias final:   b = {round(res['b'], 4)}")

---
## 📈 Graficar Convergencia

Visualizamos cómo disminuyen los errores a lo largo de las épocas.

In [ ]:
graficar_convergencia(res['history'], "Convergencia — Perceptrón Sensor")

---
## 🗺️ Frontera de Decisión

Graficamos la recta que separa las clases en el espacio de 2 features.

In [ ]:
graficar_frontera_decision(res['W'], res['b'], X, y, "Frontera — Temperatura vs Luz")

---
## ✅ Evaluar Precisión en Datos de Prueba

Separamos datos de entrenamiento y prueba para medir el rendimiento real del modelo.

In [ ]:
# Dividir en train/test
np.random.seed(123)
indices = np.random.permutation(N)
split = int(N * 0.8)
train_idx, test_idx = indices[:split], indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"Entrenamiento: {len(X_train)} muestras")
print(f"Prueba:        {len(X_test)} muestras")

# Re-entrenar con datos de entrenamiento
res_final = perceptron_entrenar(X_train, y_train, lr=0.001, epochs=100, verbose=False)

preds_train = perceptron_binario(res_final['W'], res_final['b'], X_train)
preds_test = perceptron_binario(res_final['W'], res_final['b'], X_test)

acc_train = np.mean(preds_train == y_train) * 100
acc_test = np.mean(preds_test == y_test) * 100

print(f"\nPrecisión entrenamiento: {acc_train:.1f}%")
print(f"Precisión prueba:        {acc_test:.1f}%")

---
## 📤 Exportar Perceptrón a C++

Generamos código C++ listo para copiar al Arduino IDE.  
Los pesos se escalan a enteros (×100) para evitar operaciones float en el microcontrolador.

In [ ]:
codigo = exportar_perceptron_a_c(
    res_final['W'], res_final['b'],
    nombre_funcion='clasificar',
    nombres_entrada=['temperatura', 'luz'],
    tipo='int'
)
print(codigo)

---
## 📟 Exportar Sketch Arduino Completo (.ino)

Generamos un archivo `.ino` completo con `setup()`, `loop()`, y la función del modelo.  
Incluye lectura de sensores analógicos, inferencia y control de LED.

In [ ]:
sketch = exportar_modelo_arduino_ino(
    res_final['W'], res_final['b'],
    nombre='SensorML',
    tipo='int'
)
print(sketch)

---
## 📦 Guardar el Sketch a Disco

Podemos guardar directamente el archivo `.ino` para abrirlo con Arduino IDE.

In [ ]:
with open("SensorML.ino", "w", encoding="utf-8") as f:
    f.write(sketch)
print("✅ Archivo SensorML.ino guardado — ábrelo con Arduino IDE")

---
## 🚀 Cómo Subir a Arduino (Paso a Paso)

1. **Abre Arduino IDE**
2. **Crea un nuevo sketch** (Archivo → Nuevo)
3. **Pega el código** generado arriba
4. **Conecta tu Arduino** por USB
5. **Selecciona placa**: Herramientas → Placa → Arduino Uno (o la tuya)
6. **Selecciona puerto**: Herramientas → Puerto → COM3 (o el que corresponda)
7. **Verifica** (✓) el código — revisa que no haya errores
8. **Sube** (→) el sketch al Arduino
9. **Abre Monitor Serial** (Herramientas → Monitor Serial) a 9600 baud

### Conexión sugerida del circuito
```
Arduino        | Sensor
───────────────┼───────────────────────
A0 (pin 14)    | LM35 (temp) — salida
A1 (pin 15)    | LDR + resistor — divisor
D13            | LED_BUILTIN (salida)
```

### Verás en el Monitor Serial:
```
🧠 TinyML listo — SensorML
Entradas: 35, 720 → Predicción: 1
Entradas: 20, 300 → Predicción: 0
...
```

---
## 🔢 Concepto: Cuantización (int vs float)

Los microcontroladores como Arduino Uno (ATmega328) no tienen FPU.  
Operar con `float` es lento y ocupa mucha memoria.

### Cuantización entera
| Tipo | Rango | Bytes | Velocidad | Uso |
|------|-------|-------|-----------|-----|
| `float` | ±3.4e38 | 4 | Lenta (software) | Prototipos |
| `int` (escalado ×100) | ±32767 | 2 | Rápida (hardware) | Producción |

### Ejemplo de cuantización
```
Peso real:     w = 0.3721
Peso escalado: W = round(0.3721 × 100) = 37

suma = 37 * temperatura + 15 * luz + (-2475)
resultado = (suma >= 0) ? 1 : 0
```

### Ventajas
- **Sin FPU**: funciona en cualquier microcontrolador
- **10× más rápido** que float en Arduino Uno
- **Menos RAM**: enteros de 16 bits vs floats de 32 bits
- **Determinismo**: misma salida en cualquier plataforma

---
## 🧪 #TODO: Ejercicios

1. **Tus propios sensores**: Conecta un LM35 (temp) y un LDR (luz) al Arduino.  
   Recolecta 100 lecturas con `detectar_puertos()` + `leer_linea()`.  
   Etiqueta manualmente cuándo el ventilador debería encenderse.  
   Entrena el perceptrón y exporta a C++.

2. **3 features**: Agrega humedad como tercer sensor.  
   El perceptrón sigue funcionando (solo cambia la dimensión de W).  
   Usa `graficar_convergencia()` para ver el progreso.

3. **Cuantización manual**: Entrena con `tipo='float'` en `exportar_perceptron_a_c`  
   y compara el tamaño del código vs `tipo='int'`.

4. **Precisión vs épocas**: Varía `epochs` entre 10 y 200 y grafica la precisión final.  
   ¿A partir de cuántas épocas se estabiliza?

5. **Compuerta AND con perceptrón**: Usa `perceptron_compuerta('AND')` y exporta a C++.  
   Verifica que el código funciona como una compuerta AND.